# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
#%pip install datasets
'''
Rule 1: Improving the position of a webpage in search results can lead to higher click-through rates (CTR). 
There is a strong correlation between visibility and user engagement, so there is an inverse relationship between average position and CTR.
CONFIRMED: Average CTR = 0.004195 when average position <= 3, 0.002434 when 3 < average position <= 10, 
0.001565 when 10 < average position <= 20, and 0.000438 when average position > 20.
The rule is confirmed because n > 300k for all 4 groups and the average CTR consistently decreases as the average position increases.

Rule 2: If the impressions on a certain month decreases, it will decrease in the subsequent month too.
If the total impressions from days T to T+29 > days T+30 to T+59, then days T+30 to T+59 > days T+60 to T+89.
FALSE: 619965 / (619965+515317) = 54.6% of the pages that declined initially continued to decline.
753442 / (753442+525524) = 58.9% of the pages that did not decline initially declined later.
Pages that declined initially have a lower probability of declining later compared to pages that did not, so the rule is not confirmed.
'''
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", streaming=True, split="train")
top_3 = {'n': 0, 'ctr': 0.0}
top_10 = {'n': 0, 'ctr': 0.0}
top_20 = {'n': 0, 'ctr': 0.0}
above_20 = {'n': 0, 'ctr': 0.0}
decline_both = 0
decline_first = 0
decline_last = 0
decline_none = 0
for item in iter(ds):
    if item['avg_position_90d'] <= 3.0:
        top_3['n'] += 1
        top_3['ctr'] += item['clicks_90d'] / item['impressions_90d']
    elif item['avg_position_90d'] <= 10.0:
        top_10['n'] += 1
        top_10['ctr'] += item['clicks_90d'] / item['impressions_90d']
    elif item['avg_position_90d'] <= 20.0:
        top_20['n'] += 1
        top_20['ctr'] += item['clicks_90d'] / item['impressions_90d']
    else:
        above_20['n'] += 1
        above_20['ctr'] += item['clicks_90d'] / item['impressions_90d']
    if item['impressions_90d'] - item['impressions_last30'] - item['impressions_prev30'] > item['impressions_prev30']:
        if item['impressions_prev30'] > item['impressions_last30']: decline_both += 1
        else: decline_first += 1
    else:
        if item['impressions_prev30'] > item['impressions_last30']: decline_last += 1
        else: decline_none += 1
top_3['ctr'] /= top_3['n']
top_10['ctr'] /= top_10['n']
top_20['ctr'] /= top_20['n']
above_20['ctr'] /= above_20['n']
print(f"Avg position <= 3: {top_3}")
print(f"3 < Avg position <= 10: {top_10}")
print(f"10 < Avg position <= 20: {top_20}")  
print(f"Avg position > 20: {above_20}")
print('Impressions day 1-30 > 31-60 > 61-90:', decline_both)
print('Impressions day 1-30 > 31-60 <= 61-90:', decline_first)
print('Impressions day 1-30 <= 31-60 > 61-90:', decline_last)
print('Impressions day 1-30 <= 31-60 <= 61-90:', decline_none)

Avg position <= 3: {'n': 405442, 'ctr': 0.004195016348257143}
3 < Avg position <= 10: {'n': 978489, 'ctr': 0.0024340403042018787}
10 < Avg position <= 20: {'n': 327380, 'ctr': 0.0015647425772110533}
Avg position > 20: {'n': 702937, 'ctr': 0.00043834422654031306}
Impressions day 1-30 > 31-60 > 61-90: 619965
Impressions day 1-30 > 31-60 <= 61-90: 515317
Impressions day 1-30 <= 31-60 > 61-90: 753442
Impressions day 1-30 <= 31-60 <= 61-90: 525524


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import csv
import os

os.makedirs("work/outputs", exist_ok=True)
results = []
FIX_CTR = POSITION_TOO_DEEP = 0

for item in ds:
    impressions = item["impressions_90d"]
    clicks = item["clicks_90d"]
    position = item["avg_position_90d"]
    ctr = clicks / impressions

    if position <= 3:
        expected_ctr = 0.004195016348257143
    elif position <= 10:
        expected_ctr = 0.0024340403042018787
    elif position <= 20:
        expected_ctr = 0.0015647425772110533
    else:
        expected_ctr = 0.00043834422654031306

    ctr_gap = (expected_ctr - ctr) / expected_ctr
    position_severity = min(position / 50, 1)

    if ctr_gap > 0.4 and (ctr_gap > position_severity or ctr > 0.001):
        reason_code = "LOW_CTR_FOR_POSITION"
        action = "FIX_CTR"
        action_score = ctr_gap * impressions**.5
        FIX_CTR += 1
    elif position_severity > 0.4 and ctr <= 0.001:
        reason_code = "POSITION_TOO_DEEP"
        action = "IMPROVE_POSITION"
        action_score = position_severity * impressions**0.5
        POSITION_TOO_DEEP += 1
    else:
        continue

    results.append({
        "content_hash_id": item["content_hash_id"],
        "query_hash_id": item["query_hash_id"],
        "avg_position_90d": position,
        "impressions_90d": impressions,
        "clicks_90d": clicks,
        "ctr": ctr,
        "expected_ctr": expected_ctr,
        "action_score": action_score,
        "reason_code": reason_code,
        "action": action
    })

results.sort(key=lambda x: x["action_score"], reverse=True)
output_path = "work/outputs/baseline_action_score.csv"
with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print(f"Saved {len(results)} rows to {output_path}")

Saved 2220564 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
for i, j in enumerate(results[:20]):
    print(f"{i+1}) Action: {j['action']}, Reason code: {j['reason_code']}, Confidence note: {j['action_score']}")
'''
When action is FIX_CTR (all of the below rows except 4 and 14), avg position value near the top or bottom of a tier might make it wrong.
Avg position = 3 and 9.9 are compared against the same expected value because they are in the same tier, but 3 should be higher than 9.9 
'''

1) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 719.1237345885617
2) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 576.0486090600341
3) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 540.7161917309302
4) Action: IMPROVE_POSITION, Reason code: POSITION_TOO_DEEP, Confidence note: 531.4564892820483
5) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 528.1950397343769
6) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 514.1974328990763
7) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 504.8269744374958
8) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 498.1877156253454
9) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 457.28000174947516
10) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 451.10309243010073
11) Action: FIX_CTR, Reason code: LOW_CTR_FOR_POSITION, Confidence note: 412.3355125883797


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.